Here we test some evaluation metrics on the different files we have in our predictions folder, so comparing how the model performance changes depending on the type of test set and the model we used. 

Intial approach: Use the given span_f1 for all files 

Taha's brain: A metric depending on the specific tag we changed so for ex. If we changed the names to names from different regions, then how evaluating on just B-PER and I-PER predictions, whereas if we changed the locations then evaluating only on the B-LOC I-LOC tags, then we have things like random strings and typos there we can just compare with a simple f1 score.

An example metric here could be f1 on that specific tag

The upside of the initial approach is it is universally comparable, easy to code (code is already there just need to make adjustments because our gold and predictions are in the same file (easy peezy))

The second approach might be better but is probably a bit harder to write the code for.

Below I will do the span f1 on the original test set

In [49]:
def readNlu(path):
    """Reads given iob2 file based on path, returns gold truth and predictions as seperate lists
    ----------
    path : str
        path to an iob2 file where the second column is the ground truth and the third column is the predictions
    
    Returns
    ----------
    annotations : list
        a list with all the ground truths where each list within is a seperate sentence
    
    predicition : list
        a list with all the predictions where each list within is a seperate sentence
    """
    annotations = []
    cur_annotation = []

    prediction = []
    cur_prediction = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()
        if line == '':
            annotations.append(cur_annotation)
            cur_annotation = []

            prediction.append(cur_prediction)
            cur_prediction = []
        elif line[0] == '#' and len(line.split(' ')) == 1:
            continue
        else:
            cur_annotation.append(line.split(' ')[1])
            cur_prediction.append(line.split(' ')[2])
    return annotations, prediction

In [50]:
an, pr = readNlu("../predictions/original_test/mono/test_conll_results_mono.iob2")

In [51]:
def read_iob2_file(path):
    """
    Read provided Universal NER iob2 file
    
    :param path: path to read from
    :returns: list with sequences of words and NER labels for each sentence
    """
    data = []
    gold_ner_tags = []
    predicted_ner_tags = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()

        if line:
            if line[0] == '#':
                continue # skip comments
            tok = line.split(' ')
            #print(tok)
            gold_ner_tags.append(tok[1])
            predicted_ner_tags.append(tok[2])
        else:
            if gold_ner_tags:  # skip empty lines
                data.append((gold_ner_tags, predicted_ner_tags))
            gold_ner_tags = []
            predicted_ner_tags = []

    # check for last one
    if gold_ner_tags != []:
        data.append((gold_ner_tags, predicted_ner_tags))
    return data

d = read_iob2_file("../predictions/original_test/mono/test_conll_results_mono.iob2")

In [52]:
len(an) == len(pr) == len(d)

True

FUNCTION CHANGED SUCCESFULLY, Time to test the whole thing

In [53]:
#all the other functions are the same

def toSpans(tags):
    # Converts a list of tags to a list of spans
    # in: ['B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O']
    # out: {'7-9:ORG', '0-2:PER'}
    spans = set()
    for beg in range(len(tags)):
        if tags[beg][0] == 'B':
            end = beg
            for end in range(beg+1, len(tags)):
                if tags[end][0] != 'I':
                    break
            spans.add(str(beg) + '-' + str(end) + ':' + tags[beg][2:])
    return spans

def getBegEnd(span):
    return [int(x) for x in span.split(':')[0].split('-')]

def getLooseOverlap(spans1, spans2):
    # returns the overlap of spans without taking the exact boundaries
    # into account. If entities overlap they also count as found.
    found = 0
    for spanIdx, span in enumerate(spans1):
        spanBeg, spanEnd = getBegEnd(span)
        label = span.split(':')[1]
        match = False
        for span2idx, span2 in enumerate(spans2):
            span2Beg, span2End = getBegEnd(span2)
            label2 = span2.split(':')[1]
            if label == label2:
                if span2Beg >= spanBeg and span2Beg <= spanEnd:
                    match = True
                if span2End <= spanEnd and span2End >= spanBeg:
                    match = True
        if match:
            found += 1
    return found

def getUnlabeled(spans1, spans2):
    # Counts the overlap in spans after removing the labels
    return len(set([x.split(':')[0] for x in spans1]).intersection([x.split(':')[0] for x in spans2]))

In [57]:
#Essentially what's at the end of span_f1.py, a strict scoring system where all tags should match, 
#a loose system where even if one of the bios tag is seen so if IT Univerisity of Copenhagen and only Univerisity is detceted
#then there is still credit given, lastly the unlabelled scoring where if an entity is matched regardless of what it is,
#credit is then given, this will be especially useful in the random strings test set

def evaluate(file_path):
    gold_ners, pred_ners = readNlu(file_path)

    tp = 0
    fp = 0
    fn = 0

    recall_loose_tp = 0
    recall_loose_fn = 0
    precision_loose_tp = 0
    precision_loose_fp = 0

    tp_ul = 0
    fp_ul = 0
    fn_ul = 0 

    for gold_ner, pred_ner in zip(gold_ners, pred_ners):
        gold_spans = toSpans(gold_ner)
        pred_spans = toSpans(pred_ner)
        overlap = len(gold_spans.intersection(pred_spans))
        tp += overlap
        fp += len(pred_spans) - overlap
        fn += len(gold_spans) - overlap
        
        overlap_ul = getUnlabeled(gold_spans, pred_spans)
        tp_ul += overlap_ul
        fp_ul += len(pred_spans) - overlap_ul
        fn_ul += len(gold_spans) - overlap_ul

        overlap_loose = getLooseOverlap(gold_spans, pred_spans)
        recall_loose_tp += overlap_loose
        recall_loose_fn += len(gold_spans) - overlap_loose

        overlap_loose = getLooseOverlap(pred_spans, gold_spans)
        precision_loose_tp += overlap_loose
        precision_loose_fp += len(pred_spans) - overlap_loose

    print(f"For file {file_path}:")
    print()
    prec = 0.0 if tp+fp == 0 else tp/(tp+fp)
    rec = 0.0 if tp+fn == 0 else tp/(tp+fn)
    print('recall:   ', rec)
    print('precision:', prec)
    f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)
    print('slot-f1:  ', f1)

    print()
    print('loose (partial overlap with same label)')
    l_prec = 0.0 if precision_loose_tp + precision_loose_fp == 0 else precision_loose_tp/(precision_loose_tp+precision_loose_fp)
    l_rec = 0.0 if recall_loose_tp+recall_loose_fn == 0 else recall_loose_tp/(recall_loose_tp+recall_loose_fn)
    print('l_recall:   ', rec)
    print('l_precision:', prec)
    l_f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)
    print('l_slot-f1:  ', f1)

    tp = tp_ul
    fp = fp_ul
    fn = fn_ul
    print()
    print('unlabeled')
    ul_prec = 0.0 if tp+fp == 0 else tp/(tp+fp)
    ul_rec = 0.0 if tp+fn == 0 else tp/(tp+fn)
    print('ul_recall:   ', rec)
    print('ul_precision:', prec)
    ul_f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)
    print('ul_slot-f1:  ', f1)

    return prec, rec, f1, l_rec, l_prec, l_f1, ul_rec, ul_prec, ul_f1

        

In [61]:
from span_f1_adjusted import evaluate
import os
import pandas as pd

directory = "../predictions"
folders = os.listdir(directory)

results_data = []

for folder in folders:
    models = os.listdir(os.path.join(directory,folder))
    for model in models:
        files = os.listdir(os.path.join(directory,folder,model))
        for file in files:
            path = os.path.join(directory,folder,model,file)
            prec, rec, f1, l_rec, l_prec, l_f1, ul_rec, ul_prec, ul_f1 = evaluate(path)

            results_data.append({
                "folder": folder,
                "model": model,
                "file": file,
                "path": path,
                "precision": prec,
                "recall": rec,
                "f1": f1,
                "loose_precision": l_prec,
                "loose_recall": l_rec,
                "loose_f1": l_f1,
                "unlabeled_precision": ul_prec,
                "unlabeled_recall": ul_rec,
                "unlabeled_f1": ul_f1
            })

df = pd.DataFrame(results_data)


For file ../predictions\gender_names\mono\female_names_test_0_results_mono.iob2:

recall:    0.6756373937677054
precision: 0.6862075166336989
slot-f1:   0.6808814345615131

loose (partial overlap with same label)
l_recall:    0.6756373937677054
l_precision: 0.6862075166336989
l_slot-f1:   0.6808814345615131

unlabeled
ul_recall:    0.6756373937677054
ul_precision: 0.6862075166336989
ul_slot-f1:   0.6808814345615131
For file ../predictions\gender_names\mono\female_names_test_1_results_mono.iob2:

recall:    0.6774079320113314
precision: 0.6877584037389898
slot-f1:   0.6825439300686825

loose (partial overlap with same label)
l_recall:    0.6774079320113314
l_precision: 0.6877584037389898
l_slot-f1:   0.6825439300686825

unlabeled
ul_recall:    0.6774079320113314
ul_precision: 0.6877584037389898
ul_slot-f1:   0.6825439300686825
For file ../predictions\gender_names\mono\female_names_test_2_results_mono.iob2:

recall:    0.675814447592068
precision: 0.6854013287843419
slot-f1:   0.68057412

In [64]:
df

,folder,model,file,path,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
0,gender_names,mono,female_names_test_0_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.686208,0.675637,0.680881,0.867830,0.859065,0.680881,0.725589,0.714412,0.680881
1,gender_names,mono,female_names_test_1_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.687758,0.677408,0.682544,0.869854,0.861013,0.682544,0.724969,0.714058,0.682544
2,gender_names,mono,female_names_test_2_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.685401,0.675814,0.680574,0.866942,0.859065,0.680574,0.725624,0.715475,0.680574
3,gender_names,mono,female_names_test_3_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.686433,0.676346,0.681352,0.868284,0.859950,0.681352,0.724528,0.713881,0.681352
4,gender_names,mono,female_names_test_4_results_mono.iob2,../predictions\gender_names\mono\female_names_...,0.684816,0.674752,0.679747,0.867565,0.858888,0.679747,0.723810,0.713173,0.679747
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,pronouns,multi,female_pronouns_test_results_multi.iob2,../predictions\pronouns\multi\female_pronouns_...,0.665714,0.577550,0.618506,0.885918,0.764873,0.618506,0.699796,0.607118,0.618506
150,pronouns,multi,male_pronouns_test_results_multi.iob2,../predictions\pronouns\multi\male_pronouns_te...,0.665714,0.577550,0.618506,0.885918,0.764873,0.618506,0.699796,0.607118,0.618506
151,pronouns,multi,neutral_pronouns_test_results_multi.iob2,../predictions\pronouns\multi\neutral_pronouns...,0.665714,0.577550,0.618506,0.885918,0.764873,0.618506,0.699796,0.607118,0.618506
152,random,mono,random_conll_results_mono.iob2,../predictions\random\mono\random_conll_result...,0.365430,0.190156,0.250146,0.507315,0.262571,0.250146,0.655325,0.341006,0.250146


In [65]:
df.to_csv("../ner_evaluation_results.csv", index=False)